# Risk/UQ Paper Track: Paper Tables and Figures

## Objective
Render publication-ready tables/figures from persisted benchmark artifacts only.


## Methodology
1. Deterministic bootstrap and run-context resolution.
2. Validate upstream contract stages (`risk_training_completed`, `uq_benchmark_completed`).
3. Run export flow and preview outputs.
4. Record completion in manifest.


## Step 1 - Bootstrap Environment


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for path in (REPO_DIR, f'{REPO_DIR}/src'):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True; restart runtime before continuing.')


## Step 2 - Required Config Block + Run Context


In [ ]:
from src.workflows import initialize_run_context, report_run_context

RUN_TAG = ''
RUN_PREFIX = 'risk_uq'
PERSIST_ROOT = '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v1'

N_SHARDS = 1
SHARD_ID = 'auto'

AUTO_RUN_MAIN_LOOP_WHEN_READY = False
RUN_MAIN_LOOP_OVERRIDE = False

RUN_MODE = 'auto'  # auto | fresh | resume
WARN_ON_CONFIG_DRIFT = True

run_context = initialize_run_context(
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    auto_run_main_loop_when_ready=bool(AUTO_RUN_MAIN_LOOP_WHEN_READY),
    run_main_loop_override=RUN_MAIN_LOOP_OVERRIDE,
    run_tag_prefix=RUN_PREFIX,
    planner_backend='latentdriver',
    planner_surprise_name='predictive_seq_w2',
    auto_generate_run_tag_if_empty=True,
    resume_mode=RUN_MODE,
    warn_on_config_drift=bool(WARN_ON_CONFIG_DRIFT),
)

cfg = run_context.cfg
search_cfg = run_context.search_cfg
RUN_TAG = run_context.run_tag
SHARD_ID = run_context.shard_id
run_prefix = run_context.run_prefix

print('run_prefix =', run_prefix)
report_run_context(run_context, display_fn=display)


## Step 3 - Contract Stage Check (Training + Benchmark Complete)


In [ ]:
from src.workflows import (
    load_notebook_contract_manifest,
    validate_notebook_contract_manifest,
    write_notebook_contract_manifest,
)

manifest = load_notebook_contract_manifest(cfg.run_prefix)
ok, reasons = validate_notebook_contract_manifest(
    manifest,
    require_quick_probe=True,
    require_preflight=True,
    required_stages=('risk_training_completed', 'uq_benchmark_completed'),
)
if not ok:
    raise RuntimeError(
        'Notebook contract prerequisites not met. Run training + benchmark notebooks first. '
        f'Reasons: {reasons}'
    )

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name='paper_tables_figures_colab',
    stage='paper_export_started',
    repo_dir='/content/waymax-simulation-experiments',
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
)
print('manifest_path =', manifest_path)


## Step 4 - Export Tables and Figures


In [ ]:
from src.workflows import export_paper_tables_and_figures

EXPORT_DIR = f"{cfg.run_prefix}_paper_exports"
paper_bundle = export_paper_tables_and_figures(
    run_prefix=cfg.run_prefix,
    output_dir=EXPORT_DIR,
)

paper_bundle.exported_paths


## Step 5 - Preview Key Outputs


In [ ]:
import pandas as pd
from pathlib import Path

export_dir = Path(paper_bundle.output_dir)
calib = pd.read_csv(export_dir / 'calibration_table.csv')
shift = pd.read_csv(export_dir / 'shift_robustness_table.csv')
tradeoff = pd.read_csv(export_dir / 'controller_tradeoff_table.csv')

display(calib.head(20))
display(shift.head(20))
display(tradeoff.head(20))


## Step 6 - Update Manifest (Paper Export Completed)


In [ ]:
manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name='paper_tables_figures_colab',
    stage='paper_export_completed',
    repo_dir='/content/waymax-simulation-experiments',
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    extra_fields={
        'export_dir': str(paper_bundle.output_dir),
        'n_exported_paths': int(len(dict(paper_bundle.exported_paths))),
    },
)
print('manifest_path =', manifest_path)
